In [42]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import  StrOutputParser
from langchain_core.runnables import RunnableParallel
import os

## **Data Extraction**

In [43]:
data = PyPDFLoader('S_Dhanush.pdf').load()

In [44]:
data1 = ""
for chunk in data:
    data1 += chunk.page_content
data = data1

In [45]:
#Step 1:
promt_template1 = ChatPromptTemplate.from_messages([
    ('system','From the resume Extract only the Skills Section'),
    ('human','resume : {data}')
])

In [46]:
promt_template2 = ChatPromptTemplate.from_messages([
    ('system',"From the Resume generate ATS Score based on Data Scientist Role"),
    ('human',"resume : {data}")
])

In [47]:
promt_template3 = ChatPromptTemplate.from_messages([
    ('system',"I need Suggestion to improve my ATS Score"),
    ('human',"resume : {data}")
])

In [48]:
#Step 2:
gemini_model = ChatGoogleGenerativeAI(
    model = 'gemini-3.7-flash',
    api_key = os.environ['Gemini_api'],
    temperature = 0
)

In [49]:
parser = StrOutputParser()

In [50]:
skill_chain = promt_template1 | gemini_model | parser
ats_chain = promt_template2 | gemini_model | parser
suggestion_chain = promt_template3 | gemini_model | parser

In [51]:
full_chain = RunnableParallel(
    skills = skill_chain,
    ats = ats_chain,
    suggestion = suggestion_chain
)

In [52]:
response = full_chain.invoke({"data": data})


In [57]:
print(response['suggestion'])

Here is a comprehensive review of your resume with actionable steps to significantly boost your ATS (Applicant Tracking System) score.

---

### 1. Critical ATS "Blockers" (Must Fix Immediately)

* **Remove Formatting Artifacts (`“‘latex`):**
  * Your resume contains stray tags like `“‘latex` and `“‘`. ATS parsers (like Workday, Taleo, Greenhouse) read these as corrupted syntax or junk text, which directly lowers your parsing compatibility score.
* **Fix Unrealistic/Future Dates:**
  * You listed:
    * *Data Science Intern — PySpiders (Mar 2026 – Jul 2026)*
    * *Data Scientist Intern — VCodez (Jul 2025 – Nov 2025)*
    * *Software Developer Intern — Gateway (Mar 2025 – Jun 2025)*
  * **ATS Issue:** Listing future employment dates causes automated rejection or parser errors. Internships should be listed in **reverse chronological order** (most recent on top) and must represent actual or ongoing timelines (e.g., *Nov 2024 – Present* or *Expected Completion: 2026* for your degree).

--